# Combined Evaluation and System Demonstration

**Owner:** Member 4 (integration, evaluation)
**Purpose:** Evaluate the three models together (face, voice, product) and test the multimodal decision logic used by `src/app.py`, including the four required scenarios.
**Inputs:** `models/*.joblib`, `outputs/metrics/*.json`, `data/processed/*.csv`
**Outputs:** the evaluation tables and the four scenario results below.

The decision logic is the one in the command-line app: face check, hidden recommendation, voice check against the **same** member, and the recommendation shown only if both checks pass.

In [1]:
import os, sys, json, warnings, random
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
random.seed(RANDOM_STATE); np.random.seed(RANDOM_STATE)

if Path.cwd().name == "notebooks":
    os.chdir("..")
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import app
from preprocess_images import _verify_notebook_bundle
from train_auth_models import verify_features
print("Repository root folder:", ROOT.name)

Repository root folder: Formative-2-machine-learning


## 1. Model evaluation table
Metrics come from each model owner's saved metrics file, all measured on held-out data:
- **Face model** (Member 2): leave-one-expression-out cross-validation.
- **Voice model** (Member 3): verification against a claimed identity with an ECAPA speaker embedding. Separate train, validation and test recordings; the threshold is tuned on validation only. The table reports the **test** split.
- **Product model** (Member 1): hold-out split grouped by customer, so no customer is in both train and test.

In [2]:
M = ROOT / "outputs" / "metrics"
face = json.load(open(M / "face_model_metrics.json"))
voice = json.load(open(M / "voice_metrics.json"))
prod = json.load(open(M / "product_model_metrics.json"))
pm = prod["grouped_split"]["metrics"][prod["best_model"]]
fc = face["cross_validation"]
vt = voice["test"]["with_augmentations"]

table = pd.DataFrame([
    {"model": "Face recognition", "classifier": "Logistic Regression on FaceNet embeddings",
     "accuracy": fc["accuracy"], "F1 (weighted)": fc["weighted_f1"], "log loss": fc["log_loss"],
     "evaluated on": f"{fc['authorized_originals_evaluated']} original photos (3-fold)"},
    {"model": "Voiceprint verification", "classifier": voice["model_type"],
     "accuracy": vt["accuracy"], "F1 (weighted)": vt["f1_weighted"], "log loss": vt["log_loss"],
     "evaluated on": f"{vt['clips']} test clips (originals + augmented copies)"},
    {"model": "Product recommendation", "classifier": prod["best_model"],
     "accuracy": pm["accuracy"], "F1 (weighted)": pm["f1_weighted"], "log loss": pm["log_loss"],
     "evaluated on": f"{pm['n_rows']} held-out rows (customer-grouped)"},
]).round(3)
table

,model,classifier,accuracy,F1 (weighted),log loss,evaluated on
0,Face recognition,Logistic Regression on FaceNet embeddings,1.000,1.000,0.006,12 original photos (3-fold)
1,Voiceprint verification,ecapa_cosine (C),0.969,0.969,0.192,32 test clips (originals + augmented copies)
2,Product recommendation,logistic_regression,0.167,0.125,2.751,30 held-out rows (customer-grouped)


**Reading the table.** The face and voice models identify the four members very well, but on very few samples. The product model scores **below the 0.20 chance level** for five categories on the customer-grouped test, so its recommendations are only illustrative: the customer features carry little signal for the product category in this small dataset (Member 1 documents the same limitation in the product metrics file).

## 2. Authentication error rates
- **FAR** (false acceptance rate): how often an unauthorized person (an outsider) is accepted.
- **FRR** (false rejection rate): how often an authorized member is rejected.
- **Mismatch rate**: how often a member's voice is accepted as a different member.

Voice rates come from the **test** split (recordings the threshold was never tuned on). They are recomputed here from the saved model and compared with the numbers Member 3 reported.

In [3]:
t = face["threshold_validation"]; f_final = face["final_unauthorized_test"]

aud = pd.read_csv("data/processed/audio_features.csv")
unauth = pd.read_csv("data/processed/audio_features_unauthorized.csv")
vb = joblib.load("models/voice_model.joblib")
feat = vb["features"]
members = sorted(vb["classes"])
print("voice method:", vb["method"], "| threshold:", vb["threshold"], "| require top-1 match:", vb["params"].get("require_top1"))

def voice_decision(row, expected):
    member, score, ok = verify_features("models/voice_model.joblib", {k: row[k] for k in feat}, expected)
    return {"member": member, "confidence": score, "accepted": bool(ok)}

test_members = aud[aud.split == "test"]
test_outsiders = unauth[unauth.split == "test"]

def rates_for(only_originals):
    mem = test_members[test_members.augmentation == "original"] if only_originals else test_members
    out = test_outsiders[test_outsiders.augmentation == "original"] if only_originals else test_outsiders
    genuine = [voice_decision(r, r.member_id)["accepted"] for _, r in mem.iterrows()]
    mismatch = [voice_decision(r, m)["accepted"] for _, r in mem.iterrows() for m in members if m != r.member_id]
    outsider = [voice_decision(r, m)["accepted"] for _, r in out.iterrows() for m in members]
    return {"FRR": 1 - np.mean(genuine), "n_genuine": len(genuine),
            "mismatch": np.mean(mismatch), "n_mismatch": len(mismatch),
            "FAR": np.mean(outsider), "n_outsider": len(outsider)}

own_orig, own_all = rates_for(True), rates_for(False)
rep_orig, rep_all = voice["test"]["originals"], voice["test"]["with_augmentations"]

rows = [
    {"model": "Face", "check": "unknown face accepted (FAR)", "result": f"{f_final['false_acceptance_rate']:.0%}", "trials": f_final["images_evaluated"]},
    {"model": "Face", "check": "member rejected (FRR, validation)", "result": f"{t['false_rejection_rate']:.0%}", "trials": t["authorized_samples"]},
]
for label, mine, rep in [("original recordings", own_orig, rep_orig), ("all clips incl. augmented", own_all, rep_all)]:
    rows += [
        {"model": "Voice", "check": f"outsider accepted (FAR), {label}", "result": f"{mine['FAR']:.1%}", "trials": mine["n_outsider"]},
        {"model": "Voice", "check": f"member rejected (FRR), {label}", "result": f"{mine['FRR']:.1%}", "trials": mine["n_genuine"]},
        {"model": "Voice", "check": f"wrong member accepted (mismatch), {label}", "result": f"{mine['mismatch']:.1%}", "trials": mine["n_mismatch"]},
    ]
rates = pd.DataFrame(rows)
rates

voice method: ecapa_cosine | threshold: 0.345 | require top-1 match: True


,model,check,result,trials
0,Face,unknown face accepted (FAR),0%,3
1,Face,"member rejected (FRR, validation)",0%,12
2,Voice,"outsider accepted (FAR), original recordings",6.2%,16
3,Voice,"member rejected (FRR), original recordings",0.0%,8
4,Voice,"wrong member accepted (mismatch), original rec...",0.0%,24
5,Voice,"outsider accepted (FAR), all clips incl. augme...",10.9%,64
6,Voice,"member rejected (FRR), all clips incl. augmented",15.6%,32
7,Voice,"wrong member accepted (mismatch), all clips in...",0.0%,96


In [4]:
# Cross-check: my recomputation must match the rates Member 3 reported in voice_metrics.json
check = pd.DataFrame([
    {"measure": "FAR (originals)", "recomputed": own_orig["FAR"], "reported": rep_orig["far"]},
    {"measure": "FRR (originals)", "recomputed": own_orig["FRR"], "reported": rep_orig["frr"]},
    {"measure": "mismatch (originals)", "recomputed": own_orig["mismatch"], "reported": rep_orig["mismatch_rate"]},
    {"measure": "FAR (all clips)", "recomputed": own_all["FAR"], "reported": rep_all["far"]},
    {"measure": "FRR (all clips)", "recomputed": own_all["FRR"], "reported": rep_all["frr"]},
    {"measure": "mismatch (all clips)", "recomputed": own_all["mismatch"], "reported": rep_all["mismatch_rate"]},
]).round(4)
check["match"] = np.isclose(check.recomputed, check.reported, atol=1e-3)
check

,measure,recomputed,reported,match
0,FAR (originals),0.0625,0.0625,True
1,FRR (originals),0.0000,0.0000,True
2,mismatch (originals),0.0000,0.0000,True
3,FAR (all clips),0.1094,0.1094,True
4,FRR (all clips),0.1562,0.1562,True
5,mismatch (all clips),0.0000,0.0000,True


In [5]:
lo, hi = rep_orig["far_95ci"]
print(f"Voice FAR on test originals: {rep_orig['far_errors']}/{rep_orig['far_trials']} = {rep_orig['far']:.1%}, 95% confidence interval {lo:.0%} to {hi:.0%}")
print("Previous voice model (before the fix): unknown-voice FAR", voice["previous_model"]["unknown_voice_far"])

Voice FAR on test originals: 1/16 = 6.2%, 95% confidence interval 0% to 30%
Previous voice model (before the fix): unknown-voice FAR 0.594


**Reading the rates.** The face model rejects the unauthorized photos and accepts the members. The new voice model is a large improvement on the earlier one (unknown-voice FAR was 59.4%): no member's voice is accepted as another member, and outsiders are accepted rarely, but not never. On the original test recordings the FAR is 1 of 16 trials, and it is higher once augmented copies are included. These are **tiny samples** (only 2 outsider speakers in the test split), so the 95% interval is wide and a FAR of 0% cannot be claimed. The threshold was chosen on validation data only; the test split and the unauthorized recordings were used only to report these numbers.

## 3. System demonstration: the four scenarios
`run_transaction` in `src/app.py` is the exact decision flow of the command-line app. Here the checks use the **stored feature rows** (FaceNet embeddings and audio features) through the same saved models and thresholds, so the notebook runs without a GPU or the raw photos. The voice clips used are from the **test** split. With the raw files, the same scenarios run with:

```
python src/app.py --face <photo.jpg> --voice <recording.wav>
```

In [6]:
img = pd.read_csv("data/processed/image_features.csv")
emb_cols = [c for c in img.columns if c.startswith("embedding_")]
face_bundle = joblib.load("models/face_recognition_bundle.joblib")

def face_from_embedding(embedding, name):
    r = _verify_notebook_bundle(face_bundle, np.asarray(embedding, float), name)
    return {"member": r.predicted_member, "confidence": r.classifier_confidence, "accepted": r.accepted}

member_photo = img[(img.member_id == "member01") & (img.augmentation == "original")].iloc[0]
rng = np.random.default_rng(RANDOM_STATE)
stranger = rng.normal(size=512); stranger /= np.linalg.norm(stranger)   # a face unlike any member

test_orig = test_members[test_members.augmentation == "original"]
voice_member01 = test_orig[test_orig.member_id == "member01"].iloc[0]
voice_member02 = test_orig[test_orig.member_id == "member02"].iloc[0]
outsider_orig = test_outsiders[test_outsiders.augmentation == "original"]
voice_stranger = outsider_orig.iloc[0]
print("Voice clips used:", voice_member01.sample_id, "|", voice_member02.sample_id, "|", voice_stranger.sample_id)

def scenario(title, embedding, voice_row):
    print("#" * 70); print(title); print("#" * 70)
    return app.run_transaction(
        "face", "voice",
        face_checker=lambda _: face_from_embedding(embedding, "photo"),
        voice_checker=lambda _, expected: voice_decision(voice_row, expected))

results = {}
results["1 authorized member"] = scenario("1. Authorized face + the same member's voice", member_photo[emb_cols], voice_member01)
print()
results["2 unauthorized face"] = scenario("2. Unauthorized face", stranger, voice_member01)
print()
results["3 unauthorized voice"] = scenario("3. Authorized face (member01) + an outsider's voice", member_photo[emb_cols], voice_stranger)
print()
results["4 voice mismatch"] = scenario("4. Authorized face (member01) + another member's voice (member02)", member_photo[emb_cols], voice_member02)

Voice clips used: member01_confirm_transaction_3_original | member02_confirm_transaction_3_original | outsider01_confirm_transaction_original
######################################################################
1. Authorized face + the same member's voice
######################################################################
=== Multimodal authentication and product recommendation ===
Face recognised as member01 (confidence 1.00).


A recommendation is ready. Voice verification is required to display it.
Voice verified as member01 (confidence 0.51).
ACCESS GRANTED. Recommended products:
  1. Books (31%)
  2. Groceries (24%)
  3. Sports (17%)

######################################################################
2. Unauthorized face
######################################################################
=== Multimodal authentication and product recommendation ===
ACCESS DENIED: face not recognised (closest match member01, confidence 0.51).

######################################################################
3. Authorized face (member01) + an outsider's voice
######################################################################
=== Multimodal authentication and product recommendation ===
Face recognised as member01 (confidence 1.00).
A recommendation is ready. Voice verification is required to display it.
ACCESS DENIED: voice sounds like member03, not member01 (confidence 0.33).

################################

In [7]:
pd.DataFrame([{"scenario": k, "outcome": v["outcome"], "reason": v["reason"]} for k, v in results.items()])

,scenario,outcome,reason
0,1 authorized member,GRANTED,face and voice verified
1,2 unauthorized face,DENIED,face not recognised
2,3 unauthorized voice,DENIED,"voice sounds like member03, not member01"
3,4 voice mismatch,DENIED,"voice sounds like member02, not member01"


**Reading the scenarios.** Scenario 1 is granted and shows the recommendation. Scenarios 2, 3 and 4 are denied, and a recommendation is never shown when either check fails. Scenario 3 uses the first outsider clip in the test split (not hand-picked); the voice FAR above shows that other outsider clips can occasionally pass.

## 4. Findings, limitations and next steps
- The multimodal logic works: the recommendation is computed after the face check, kept hidden, and displayed only after the voice matches the **same** member.
- **Face:** strong separation, but only 3 original photos per member and one unknown person were available.
- **Voice:** much better than the first model (ECAPA speaker embedding, claimed-identity verification, threshold tuned on validation data). Outsider acceptance is low but not zero, and the test set is small (2 outsider speakers, 8 genuine recordings), so the error rates have wide confidence intervals.
- **Product:** below chance on a customer-grouped test, so recommendations should not be treated as reliable.
- **Demo assumption:** there is no member-to-customer table, so each member is linked to one customer from the merged data (member01 to the first customer id, and so on).
- **Command-line app:** the voice step needs the ECAPA embedding, so SpeechBrain and PyTorch must be installed (see `requirements.txt`); the app prints a clear message if they are missing.
- **Privacy:** raw photos and recordings are biometric data and need participant consent.